# 07 - Model Training

Objective

This notebook prepares the modeling datasets, trains baseline and tuned candidate models, and selects the candidate final model for downstream evaluation.

The workflow preserves temporal ordering, prevents target leakage, compares Logistic Regression, Random Forest, and XGBoost, and persists modeling checkpoints for the evaluation and explainability notebooks.

Holdout testing, calibration analysis, fitted-model artifact saving, and model explainability are performed in Notebooks 08 and 09.

#### Load project configuration


In [0]:
import importlib.util
from pathlib import Path

_bootstrap = Path.cwd() / "notebooks" / "import_path.py"
if not _bootstrap.is_file():
    _bootstrap = Path.cwd() / "import_path.py"
_spec = importlib.util.spec_from_file_location("import_path", _bootstrap)
_ip = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_ip)

# Load the project configuration

from config import project_config as cfg

print("Project configuration loaded successfully.")


#### Load and validate the feature dataset

The model-training process begins by loading the managed `flights_features` Delta table produced by the Feature Engineering notebook.

Before splitting or modelling, the dataset is validated to confirm that:

- The required Unity Catalog table exists
- The target variable is available
- The flight date is stored as a valid date
- All required schedule-time predictors are present
- The dataset contains records suitable for chronological splitting


In [0]:
from __future__ import annotations

import importlib.util
from pathlib import Path

_bootstrap = Path.cwd() / "notebooks" / "import_path.py"
if not _bootstrap.is_file():
    _bootstrap = Path.cwd() / "import_path.py"
_spec = importlib.util.spec_from_file_location("import_path", _bootstrap)
_ip = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_ip)

from config import project_config as cfg
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql import types as T



FEATURE_TABLE = cfg.FEATURES_TABLE
TARGET_COLUMN = cfg.TARGET_COLUMN
DATE_COLUMN = cfg.FLIGHT_DATE_COLUMN


def require_table(table_name: str) -> None:
    """Raise an error when a required Unity Catalog table is unavailable."""
    if not spark.catalog.tableExists(table_name):
        raise RuntimeError(
            f"Required table '{table_name}' was not found. "
            "Run the feature-engineering notebook (06) before continuing."
        )


require_table(FEATURE_TABLE)

df_features: DataFrame = spark.table(FEATURE_TABLE)

required_columns = cfg.MODEL_TRAINING_REQUIRED_COLUMNS

missing_columns = sorted(required_columns - set(df_features.columns))

if missing_columns:
    raise ValueError(
        "Model-training validation failed. "
        f"Missing required columns: {missing_columns}"
    )

feature_row_count = df_features.count()
feature_column_count = len(df_features.columns)

date_type = df_features.schema[DATE_COLUMN].dataType

if not isinstance(date_type, T.DateType):
    raise TypeError(
        f"{DATE_COLUMN} must be a Spark date column, "
        f"but found {date_type.simpleString()}."
    )

print("Feature dataset loaded and validated successfully.")
print(f"Source table: {FEATURE_TABLE}")
print(f"Total records: {feature_row_count:,}")
print(f"Total columns: {feature_column_count}")
print(f"Prediction target: {TARGET_COLUMN}")
print(f"Date column type: {date_type.simpleString()}")

#### Date range and target distribution

Before defining the chronological training, validation, and test periods, the feature dataset is examined to confirm its available date range and the distribution of the binary target variable.

This review supports two important modelling decisions:

- Selecting non-overlapping chronological split periods
- Assessing whether the delayed and on-time classes are imbalanced

No records are modified during this analysis.


In [0]:
dataset_profile = (
    df_features
    .select(
        F.min("FL_DATE").alias("MIN_FL_DATE"),
        F.max("FL_DATE").alias("MAX_FL_DATE"),
        F.count("*").alias("TOTAL_RECORDS"),
        F.sum(
            F.when(F.col("ARR_DEL15") == 0, 1).otherwise(0)
        ).alias("ON_TIME_RECORDS"),
        F.sum(
            F.when(F.col("ARR_DEL15") == 1, 1).otherwise(0)
        ).alias("DELAYED_RECORDS"),
    )
    .withColumn(
        "ON_TIME_PERCENTAGE",
        F.round(
            F.col("ON_TIME_RECORDS") / F.col("TOTAL_RECORDS") * 100,
            4,
        ),
    )
    .withColumn(
        "DELAYED_PERCENTAGE",
        F.round(
            F.col("DELAYED_RECORDS") / F.col("TOTAL_RECORDS") * 100,
            4,
        ),
    )
)

display(dataset_profile)

#### Create chronological train, validation, and test splits

The dataset is divided chronologically rather than randomly because the model is intended to predict future flight-delay risk from historical observations.

The split periods are defined as follows:

- **Training period:** January 1, 2025 to August 31, 2025
- **Validation period:** September 1, 2025 to October 31, 2025
- **Test period:** November 1, 2025 to December 31, 2025

This design ensures that later flight outcomes are not used to train models evaluated on earlier periods. The validation dataset will support model and hyperparameter selection, while the test dataset will remain untouched until final evaluation.


In [0]:
TRAIN_END_DATE = cfg.TRAIN_END_DATE
VALIDATION_START_DATE = cfg.VALIDATION_START_DATE
VALIDATION_END_DATE = cfg.VALIDATION_END_DATE
TEST_START_DATE = cfg.TEST_START_DATE

df_train = df_features.filter(
    F.col("FL_DATE") <= F.to_date(F.lit(TRAIN_END_DATE))
)

df_validation = df_features.filter(
    (F.col("FL_DATE") >= F.to_date(F.lit(VALIDATION_START_DATE)))
    & (F.col("FL_DATE") <= F.to_date(F.lit(VALIDATION_END_DATE)))
)

df_test = df_features.filter(
    F.col("FL_DATE") >= F.to_date(F.lit(TEST_START_DATE))
)

split_summary = (
    df_train.select(
        F.lit("TRAIN").alias("DATASET"),
        F.min("FL_DATE").alias("MIN_DATE"),
        F.max("FL_DATE").alias("MAX_DATE"),
        F.count("*").alias("TOTAL_RECORDS"),
        F.avg(F.col("ARR_DEL15").cast("double")).alias("DELAY_RATE"),
    )
    .unionByName(
        df_validation.select(
            F.lit("VALIDATION").alias("DATASET"),
            F.min("FL_DATE").alias("MIN_DATE"),
            F.max("FL_DATE").alias("MAX_DATE"),
            F.count("*").alias("TOTAL_RECORDS"),
            F.avg(F.col("ARR_DEL15").cast("double")).alias("DELAY_RATE"),
        )
    )
    .unionByName(
        df_test.select(
            F.lit("TEST").alias("DATASET"),
            F.min("FL_DATE").alias("MIN_DATE"),
            F.max("FL_DATE").alias("MAX_DATE"),
            F.count("*").alias("TOTAL_RECORDS"),
            F.avg(F.col("ARR_DEL15").cast("double")).alias("DELAY_RATE"),
        )
    )
    .withColumn(
        "DELAY_PERCENTAGE",
        F.round(F.col("DELAY_RATE") * 100, 4),
    )
    .drop("DELAY_RATE")
)

display(split_summary)

#### Split validation summary

The chronological split produced three non-overlapping datasets whose combined record count matches the complete feature dataset.

The target distribution varies across the periods:

- The training period has a delay rate of approximately 22.91%.
- The validation period has a lower delay rate of approximately 18.55%.
- The test period has a higher delay rate of approximately 23.71%.

This variation reflects temporal changes in airline operations and confirms the importance of evaluating the model on future periods rather than using a random split.


#### Engineer Strictly Leakage-Safe Historical Features

Historical delay-rate features summarize prior airline, airport, and route performance. For every training date, both the entity history and its smoothing prior are calculated from **strictly earlier dates only**. Outcomes from the current date and all later dates are excluded.

The initial value `0.5` is used only when the dataset contains no earlier observations. Once earlier flights exist, the cumulative delay rate through the preceding date becomes the date-specific smoothing prior. This removes the former January–August global fallback that allowed early flights to indirectly receive information from later months.


In [0]:
from pyspark.sql.window import Window

SMOOTHING_STRENGTH = 100.0
INITIAL_DELAY_PRIOR = 0.5

# A causal global prior for each date, based only on previous dates.
global_daily_stats = (
    df_train
    .groupBy("FL_DATE")
    .agg(
        F.count("*").alias("GLOBAL_DAILY_FLIGHTS"),
        F.sum(F.col(TARGET_COLUMN).cast("long")).alias(
            "GLOBAL_DAILY_DELAYS"
        ),
    )
)

global_history_window = (
    Window
    .orderBy(F.col("FL_DATE").cast("timestamp").cast("long"))
    .rowsBetween(Window.unboundedPreceding, -1)
)

date_specific_priors = (
    global_daily_stats
    .withColumn(
        "GLOBAL_PRIOR_FLIGHTS",
        F.sum("GLOBAL_DAILY_FLIGHTS").over(global_history_window),
    )
    .withColumn(
        "GLOBAL_PRIOR_DELAYS",
        F.sum("GLOBAL_DAILY_DELAYS").over(global_history_window),
    )
    .withColumn(
        "DATE_PRIOR_DELAY_RATE",
        F.when(
            F.col("GLOBAL_PRIOR_FLIGHTS").isNull()
            | (F.col("GLOBAL_PRIOR_FLIGHTS") == 0),
            F.lit(INITIAL_DELAY_PRIOR),
        ).otherwise(
            F.col("GLOBAL_PRIOR_DELAYS").cast("double")
            / F.col("GLOBAL_PRIOR_FLIGHTS").cast("double")
        ),
    )
    .select("FL_DATE", "DATE_PRIOR_DELAY_RATE")
)


def causal_entity_history(
    source_df,
    entity_columns,
    output_prefix,
):
    # Create a smoothed entity rate using only earlier dates.
    daily_flights = f"{output_prefix}_DAILY_FLIGHTS"
    daily_delays = f"{output_prefix}_DAILY_DELAYS"
    prior_flights = f"{output_prefix}_PRIOR_FLIGHTS"
    prior_delays = f"{output_prefix}_PRIOR_DELAYS"
    rate_column = f"{output_prefix}_HIST_DELAY_RATE"

    daily = (
        source_df
        .groupBy(*entity_columns, "FL_DATE")
        .agg(
            F.count("*").alias(daily_flights),
            F.sum(F.col(TARGET_COLUMN).cast("long")).alias(daily_delays),
        )
    )
    history_window = (
        Window
        .partitionBy(*entity_columns)
        .orderBy(F.col("FL_DATE").cast("timestamp").cast("long"))
        .rowsBetween(Window.unboundedPreceding, -1)
    )

    return (
        daily
        .withColumn(prior_flights, F.sum(daily_flights).over(history_window))
        .withColumn(prior_delays, F.sum(daily_delays).over(history_window))
        .join(date_specific_priors, on="FL_DATE", how="left")
        .withColumn(
            rate_column,
            (
                F.coalesce(F.col(prior_delays).cast("double"), F.lit(0.0))
                + F.lit(SMOOTHING_STRENGTH)
                * F.col("DATE_PRIOR_DELAY_RATE")
            )
            /
            (
                F.coalesce(F.col(prior_flights).cast("double"), F.lit(0.0))
                + F.lit(SMOOTHING_STRENGTH)
            ),
        )
        .select(
            *entity_columns,
            "FL_DATE",
            prior_flights,
            prior_delays,
            rate_column,
        )
    )

print("Strictly causal date-specific smoothing priors created.")


#### Historical Airline Delay Rate

`AIRLINE_HIST_DELAY_RATE` uses the airline's outcomes from earlier dates and the causal date-specific prior. The current date and later months cannot influence the value.


In [0]:
airline_history_features = causal_entity_history(
    df_train,
    ["OP_UNIQUE_CARRIER"],
    "AIRLINE",
)

df_train_hist = (
    df_train
    .join(
        airline_history_features,
        on=["OP_UNIQUE_CARRIER", "FL_DATE"],
        how="left",
    )
    .join(date_specific_priors, on="FL_DATE", how="left")
    .withColumn(
        "AIRLINE_HIST_DELAY_RATE",
        F.coalesce(
            F.col("AIRLINE_HIST_DELAY_RATE"),
            F.col("DATE_PRIOR_DELAY_RATE"),
            F.lit(INITIAL_DELAY_PRIOR),
        ),
    )
    .drop("DATE_PRIOR_DELAY_RATE")
)

print(f"Training rows after causal airline join: {df_train_hist.count():,}")


#### Historical Origin-Airport Delay Rate

`ORIGIN_HIST_DELAY_RATE` uses only flights from earlier dates at the same origin, smoothed toward the cumulative delay rate available before the current date.


In [0]:
origin_history_features = causal_entity_history(
    df_train,
    ["ORIGIN"],
    "ORIGIN",
)

df_train_hist = (
    df_train_hist
    .join(origin_history_features, on=["ORIGIN", "FL_DATE"], how="left")
    .join(date_specific_priors, on="FL_DATE", how="left")
    .withColumn(
        "ORIGIN_HIST_DELAY_RATE",
        F.coalesce(
            F.col("ORIGIN_HIST_DELAY_RATE"),
            F.col("DATE_PRIOR_DELAY_RATE"),
            F.lit(INITIAL_DELAY_PRIOR),
        ),
    )
    .drop("DATE_PRIOR_DELAY_RATE")
)


#### Historical Destination-Airport Delay Rate

`DEST_HIST_DELAY_RATE` uses only flights from earlier dates with the same destination, with the same strictly causal smoothing rule.


In [0]:
dest_history_features = causal_entity_history(
    df_train,
    ["DEST"],
    "DEST",
)

df_train_hist = (
    df_train_hist
    .join(dest_history_features, on=["DEST", "FL_DATE"], how="left")
    .join(date_specific_priors, on="FL_DATE", how="left")
    .withColumn(
        "DEST_HIST_DELAY_RATE",
        F.coalesce(
            F.col("DEST_HIST_DELAY_RATE"),
            F.col("DATE_PRIOR_DELAY_RATE"),
            F.lit(INITIAL_DELAY_PRIOR),
        ),
    )
    .drop("DATE_PRIOR_DELAY_RATE")
)


#### Historical Route Delay Rate

`ROUTE_HIST_DELAY_RATE` uses only earlier dates on the same origin–destination route. Sparse routes are smoothed toward the causal prior available before that flight date.


In [0]:
route_history_features = causal_entity_history(
    df_train,
    ["ORIGIN", "DEST"],
    "ROUTE",
)

df_train_hist = (
    df_train_hist
    .join(
        route_history_features,
        on=["ORIGIN", "DEST", "FL_DATE"],
        how="left",
    )
    .join(date_specific_priors, on="FL_DATE", how="left")
    .withColumn(
        "ROUTE_HIST_DELAY_RATE",
        F.coalesce(
            F.col("ROUTE_HIST_DELAY_RATE"),
            F.col("DATE_PRIOR_DELAY_RATE"),
            F.lit(INITIAL_DELAY_PRIOR),
        ),
    )
    .drop("DATE_PRIOR_DELAY_RATE")
)

print("All training historical rates use strictly earlier dates.")


#### Apply Development-Period History to Later Validation and Test Data

September–October validation and November–December test records occur after the January–August development period. Their airline, airport, and route mappings may therefore use the complete development history without temporal leakage. Unseen categories fall back to the January–August development delay rate.


In [0]:
# Safe fallback for periods after the January–August development window.
global_training_delay_rate = (
    df_train
    .select(F.avg(F.col(TARGET_COLUMN).cast('double')).alias('RATE'))
    .first()['RATE']
)

# -------------------------------------------------------
# Create historical mappings from training data only
# -------------------------------------------------------

airline_training_map = (
    df_train
    .groupBy("OP_UNIQUE_CARRIER")
    .agg(
        F.count("*").alias("AIRLINE_TRAIN_FLIGHTS"),
        F.sum(F.col("ARR_DEL15").cast("long")).alias(
            "AIRLINE_TRAIN_DELAYS"
        ),
    )
    .withColumn(
        "AIRLINE_HIST_DELAY_RATE",
        (
            F.col("AIRLINE_TRAIN_DELAYS").cast("double")
            + F.lit(SMOOTHING_STRENGTH * global_training_delay_rate)
        )
        /
        (
            F.col("AIRLINE_TRAIN_FLIGHTS").cast("double")
            + F.lit(SMOOTHING_STRENGTH)
        ),
    )
    .select(
        "OP_UNIQUE_CARRIER",
        "AIRLINE_HIST_DELAY_RATE",
    )
)

origin_training_map = (
    df_train
    .groupBy("ORIGIN")
    .agg(
        F.count("*").alias("ORIGIN_TRAIN_FLIGHTS"),
        F.sum(F.col("ARR_DEL15").cast("long")).alias(
            "ORIGIN_TRAIN_DELAYS"
        ),
    )
    .withColumn(
        "ORIGIN_HIST_DELAY_RATE",
        (
            F.col("ORIGIN_TRAIN_DELAYS").cast("double")
            + F.lit(SMOOTHING_STRENGTH * global_training_delay_rate)
        )
        /
        (
            F.col("ORIGIN_TRAIN_FLIGHTS").cast("double")
            + F.lit(SMOOTHING_STRENGTH)
        ),
    )
    .select(
        "ORIGIN",
        "ORIGIN_HIST_DELAY_RATE",
    )
)

dest_training_map = (
    df_train
    .groupBy("DEST")
    .agg(
        F.count("*").alias("DEST_TRAIN_FLIGHTS"),
        F.sum(F.col("ARR_DEL15").cast("long")).alias(
            "DEST_TRAIN_DELAYS"
        ),
    )
    .withColumn(
        "DEST_HIST_DELAY_RATE",
        (
            F.col("DEST_TRAIN_DELAYS").cast("double")
            + F.lit(SMOOTHING_STRENGTH * global_training_delay_rate)
        )
        /
        (
            F.col("DEST_TRAIN_FLIGHTS").cast("double")
            + F.lit(SMOOTHING_STRENGTH)
        ),
    )
    .select(
        "DEST",
        "DEST_HIST_DELAY_RATE",
    )
)

route_training_map = (
    df_train
    .groupBy("ORIGIN", "DEST")
    .agg(
        F.count("*").alias("ROUTE_TRAIN_FLIGHTS"),
        F.sum(F.col("ARR_DEL15").cast("long")).alias(
            "ROUTE_TRAIN_DELAYS"
        ),
    )
    .withColumn(
        "ROUTE_HIST_DELAY_RATE",
        (
            F.col("ROUTE_TRAIN_DELAYS").cast("double")
            + F.lit(SMOOTHING_STRENGTH * global_training_delay_rate)
        )
        /
        (
            F.col("ROUTE_TRAIN_FLIGHTS").cast("double")
            + F.lit(SMOOTHING_STRENGTH)
        ),
    )
    .select(
        "ORIGIN",
        "DEST",
        "ROUTE_HIST_DELAY_RATE",
    )
)


def attach_training_history(dataset: DataFrame) -> DataFrame:
    """Attach historical rates calculated exclusively from training data."""
    return (
        dataset
        .join(
            airline_training_map,
            on="OP_UNIQUE_CARRIER",
            how="left",
        )
        .join(
            origin_training_map,
            on="ORIGIN",
            how="left",
        )
        .join(
            dest_training_map,
            on="DEST",
            how="left",
        )
        .join(
            route_training_map,
            on=["ORIGIN", "DEST"],
            how="left",
        )
        .fillna(
            {
                "AIRLINE_HIST_DELAY_RATE": global_training_delay_rate,
                "ORIGIN_HIST_DELAY_RATE": global_training_delay_rate,
                "DEST_HIST_DELAY_RATE": global_training_delay_rate,
                "ROUTE_HIST_DELAY_RATE": global_training_delay_rate,
            }
        )
    )


df_validation_hist = attach_training_history(df_validation)
df_test_hist = attach_training_history(df_test)

print(
    f"Validation rows after historical joins: "
    f"{df_validation_hist.count():,}"
)
print(
    f"Test rows after historical joins: "
    f"{df_test_hist.count():,}"
)

display(
    df_validation_hist
    .select(
        "FL_DATE",
        "OP_UNIQUE_CARRIER",
        "ORIGIN",
        "DEST",
        "AIRLINE_HIST_DELAY_RATE",
        "ORIGIN_HIST_DELAY_RATE",
        "DEST_HIST_DELAY_RATE",
        "ROUTE_HIST_DELAY_RATE",
        "ARR_DEL15",
    )
    .limit(20)
)


#### Validate historical performance features

The historical feature datasets are validated before categorical encoding and model training.

This check confirms that:

- Record counts remain unchanged after historical-feature joins
- No historical delay-rate features contain missing values
- All historical rates fall within the valid probability range of 0 to 1
- Training, validation, and test datasets contain the same historical feature columns


In [0]:
HISTORICAL_RATE_COLUMNS = list(cfg.MODEL_HISTORICAL_RATE_COLUMNS)

historical_validation_rows = []

for dataset_name, dataset in [
    ("TRAIN", df_train_hist),
    ("VALIDATION", df_validation_hist),
    ("TEST", df_test_hist),
]:
    summary_row = (
        dataset
        .select(
            F.lit(dataset_name).alias("DATASET"),
            F.count("*").alias("TOTAL_RECORDS"),
            *[
                F.sum(
                    F.when(F.col(column_name).isNull(), 1).otherwise(0)
                ).alias(f"{column_name}_NULLS")
                for column_name in HISTORICAL_RATE_COLUMNS
            ],
            *[
                F.sum(
                    F.when(
                        (F.col(column_name) < 0)
                        | (F.col(column_name) > 1),
                        1,
                    ).otherwise(0)
                ).alias(f"{column_name}_OUT_OF_RANGE")
                for column_name in HISTORICAL_RATE_COLUMNS
            ],
        )
    )

    historical_validation_rows.append(summary_row)

historical_validation_summary = historical_validation_rows[0]

for summary_row in historical_validation_rows[1:]:
    historical_validation_summary = (
        historical_validation_summary.unionByName(summary_row)
    )

display(historical_validation_summary)


#### Prepare Raw Predictor Columns for Standard Python

The leakage-safe historical training, validation, and test frames retain their original categorical and numerical predictor columns. Spark is used only to prepare, filter, and sample these large datasets.

Categorical encoding is deliberately deferred until after each chronological training fold has been sampled. A scikit-learn `ColumnTransformer` is fitted only on that fold's training observations and then applied to its later validation month. This keeps category learning inside the training boundary and avoids preprocessing leakage.


In [0]:
import importlib.util
from pathlib import Path

_bootstrap = Path.cwd() / "notebooks" / "import_path.py"
if not _bootstrap.is_file():
    _bootstrap = Path.cwd() / "import_path.py"
_spec = importlib.util.spec_from_file_location("import_path", _bootstrap)
_ip = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_ip)

from utils.model_training import prepare_hist_modeling_frame

CATEGORICAL_COLUMNS = list(cfg.MODEL_CATEGORICAL_COLUMNS)
NUMERICAL_COLUMNS = list(cfg.MODEL_NUMERICAL_COLUMNS)
MODEL_INPUT_COLUMNS = list(cfg.MODEL_INPUT_COLUMNS)

df_train_hist = prepare_hist_modeling_frame(df_train_hist)
df_validation_hist = prepare_hist_modeling_frame(df_validation_hist)
df_test_hist = prepare_hist_modeling_frame(df_test_hist)

df_train_prepared = df_train_hist.select(
    "FL_DATE", *MODEL_INPUT_COLUMNS, TARGET_COLUMN
)
df_validation_prepared = df_validation_hist.select(
    "FL_DATE", *MODEL_INPUT_COLUMNS, TARGET_COLUMN
)
df_test_prepared = df_test_hist.select(
    "FL_DATE", *MODEL_INPUT_COLUMNS, TARGET_COLUMN
)

print("Raw predictor datasets prepared for scikit-learn preprocessing.")
print(f"Categorical predictors: {len(CATEGORICAL_COLUMNS)}")
print(f"Numerical predictors: {len(NUMERICAL_COLUMNS)}")
print(f"Total predictors: {len(MODEL_INPUT_COLUMNS)}")


#### Validate the Raw Model Datasets

The raw predictor schema is validated before sampling. Each dataset must contain the flight date, target, and exactly the predictor columns expected by the standard-Python preprocessing pipeline.

No categorical mappings or numerical imputation values are learned at this stage.


In [0]:
required_model_columns = {
    "FL_DATE", TARGET_COLUMN, *MODEL_INPUT_COLUMNS
}

for dataframe_name, dataframe in {
    "df_train_prepared": df_train_prepared,
    "df_validation_prepared": df_validation_prepared,
    "df_test_prepared": df_test_prepared,
}.items():
    missing_columns = sorted(
        required_model_columns - set(dataframe.columns)
    )
    if missing_columns:
        raise ValueError(
            f"{dataframe_name} is missing columns: {missing_columns}"
        )
    if dataframe.limit(1).count() == 0:
        raise ValueError(f"{dataframe_name} contains no rows.")
    print(f"{dataframe_name} schema validated successfully.")


#### Standard-Python Preprocessing Boundary

After bounded sampling, raw Spark rows are collected into pandas. Scikit-learn then performs median numerical imputation, most-frequent categorical imputation, sparse one-hot encoding with unknown-category handling, and sparse-safe numerical scaling.

Every preprocessing transformer is fitted on training rows only. Logistic Regression, Random Forest, and XGBoost receive the same transformed sparse matrices within each fold.


In [0]:
print("Standard-Python preprocessing inputs are ready.")
print(f"Training rows: {df_train_prepared.count():,}")
print(f"Validation rows: {df_validation_prepared.count():,}")
print(f"Test rows: {df_test_prepared.count():,}")

display(
    df_train_prepared.select(
        "FL_DATE",
        *MODEL_INPUT_COLUMNS,
        TARGET_COLUMN,
    ).limit(10)
)


## Standard-Python Candidate Modeling

Logistic Regression, Random Forest, and XGBoost are implemented using standard Python libraries rather than Spark ML. Spark remains responsible only for loading the large Delta tables, applying the leakage-safe feature engineering workflow, and creating bounded samples that can be safely collected by the Databricks Free Edition driver.

The three algorithms receive the same SciPy sparse feature matrices, chronological folds, validation observations, evaluation metrics, and selection policy. This shared design removes implementation differences that could otherwise make the comparison difficult to interpret.

The candidate estimators are:

- `sklearn.linear_model.LogisticRegression`
- `sklearn.ensemble.RandomForestClassifier`
- `xgboost.XGBClassifier`


### Install Standard-Python Modeling Dependencies

Databricks Free Edition may not include XGBoost in a new Python session. The following installation cell runs before any modeling-library imports so that **Run All** can prepare the required standard-Python environment automatically.

The XGBoost version is pinned for reproducibility. Databricks may display routine package-installation messages while this cell runs.


In [0]:
%pip install --quiet xgboost==2.1.4 scipy scikit-learn


In [0]:
try:
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt

    from scipy.sparse import csr_matrix
    from sklearn.compose import ColumnTransformer
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.calibration import calibration_curve
    from sklearn.impute import SimpleImputer
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import (
        accuracy_score,
        average_precision_score,
        brier_score_loss,
        confusion_matrix,
        precision_recall_fscore_support,
        roc_auc_score,
    )
    from sklearn.model_selection import ParameterGrid
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import OneHotEncoder, StandardScaler
    from xgboost import XGBClassifier
except ImportError as error:
    raise ImportError(
        "Notebook 07 requires NumPy, SciPy, scikit-learn, and XGBoost. "
        "Run `%pip install xgboost==2.1.4 scipy scikit-learn`, restart "
        "Python once, and rerun the notebook from the beginning."
    ) from error

import ast
import time

from pyspark.sql import functions as F

RANDOM_SEED = cfg.RANDOM_SEED
LOCAL_TRAIN_MAX_ROWS = 200_000
LOCAL_VALIDATION_MAX_ROWS = 50_000

print("Standard-Python modeling libraries loaded successfully.")
print(f"Maximum local training rows per fold: {LOCAL_TRAIN_MAX_ROWS:,}")
print(
    "Maximum local validation rows per fold: "
    f"{LOCAL_VALIDATION_MAX_ROWS:,}"
)


## Reproducible Standard-Python Preprocessing Utilities

Bounded raw Spark samples are collected into pandas and transformed by scikit-learn. Categorical variables are one-hot encoded into a sparse matrix; numerical variables are imputed and scaled without centering so the combined representation remains sparse.

Sampling is uniform and reproducible. It does not rebalance the validation data. Consequently, the validation samples retain the natural proportion of delayed and on-time flights.


In [0]:
def bounded_uniform_sample(
    dataframe,
    maximum_rows,
    *,
    seed,
):
    """Return a reproducible uniform sample with at most maximum_rows."""
    row_count = dataframe.count()

    if row_count == 0:
        raise ValueError("Cannot sample an empty Spark DataFrame.")

    if row_count <= maximum_rows:
        return dataframe

    sampling_fraction = min(
        1.0,
        (maximum_rows * 1.10) / row_count,
    )

    return (
        dataframe
        .sample(
            withReplacement=False,
            fraction=sampling_fraction,
            seed=seed,
        )
        .limit(maximum_rows)
    )


def spark_sample_to_pandas(dataframe):
    """Collect one bounded raw Spark sample into pandas."""
    local_frame = dataframe.select(
        *MODEL_INPUT_COLUMNS,
        TARGET_COLUMN,
    ).toPandas()
    if local_frame.empty:
        raise ValueError("The local modeling sample contains zero rows.")

    for column_name in CATEGORICAL_COLUMNS:
        categorical_values = local_frame[column_name].astype(
            "object"
        )
        local_frame[column_name] = categorical_values.where(
            pd.notna(categorical_values),
            np.nan,
        )
    for column_name in NUMERICAL_COLUMNS:
        local_frame[column_name] = pd.to_numeric(
            local_frame[column_name], errors="coerce"
        )
    return local_frame


def build_sklearn_preprocessor():
    """Build a fresh transformer fitted only on training rows."""
    categorical_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=True,
                    dtype=np.float32,
                ),
            ),
        ]
    )
    numerical_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler(with_mean=False)),
        ]
    )
    return ColumnTransformer(
        transformers=[
            (
                "categorical",
                categorical_pipeline,
                CATEGORICAL_COLUMNS,
            ),
            (
                "numerical",
                numerical_pipeline,
                NUMERICAL_COLUMNS,
            ),
        ],
        sparse_threshold=1.0,
    )


def prepare_training_validation_matrices(
    training_dataframe,
    validation_dataframe,
):
    """Fit preprocessing on training and transform later validation."""
    training_local = spark_sample_to_pandas(training_dataframe)
    validation_local = spark_sample_to_pandas(validation_dataframe)

    preprocessor = build_sklearn_preprocessor()
    X_training = csr_matrix(
        preprocessor.fit_transform(
            training_local[MODEL_INPUT_COLUMNS]
        ),
        dtype=np.float32,
    )
    X_validation = csr_matrix(
        preprocessor.transform(
            validation_local[MODEL_INPUT_COLUMNS]
        ),
        dtype=np.float32,
    )

    y_training = training_local[TARGET_COLUMN].to_numpy(dtype=np.int8)
    y_validation = validation_local[TARGET_COLUMN].to_numpy(dtype=np.int8)

    return (
        preprocessor,
        X_training,
        y_training,
        X_validation,
        y_validation,
    )


def evaluate_python_classifier(model, X_validation, y_validation):
    """Evaluate a fitted binary classifier using the shared metrics."""
    predictions = model.predict(X_validation).astype(np.int8)
    probabilities = model.predict_proba(X_validation)[:, 1]

    top_count = max(1, int(np.ceil(len(y_validation) * 0.10)))
    top_indices = np.argsort(probabilities)[-top_count:]
    total_delays = int(np.sum(y_validation == 1))
    top_delays = int(np.sum(y_validation[top_indices] == 1))
    overall_delay_rate = float(np.mean(y_validation))
    top_delay_rate = float(np.mean(y_validation[top_indices]))

    weighted_precision, weighted_recall, weighted_f1, _ = (
        precision_recall_fscore_support(
            y_validation,
            predictions,
            average="weighted",
            zero_division=0,
        )
    )

    delay_precision, delay_recall, delay_f1, _ = (
        precision_recall_fscore_support(
            y_validation,
            predictions,
            average="binary",
            pos_label=1,
            zero_division=0,
        )
    )

    return {
        "ACCURACY": float(
            accuracy_score(y_validation, predictions)
        ),
        "PRECISION": float(weighted_precision),
        "RECALL": float(weighted_recall),
        "F1_SCORE": float(weighted_f1),
        "ROC_AUC": float(
            roc_auc_score(y_validation, probabilities)
        ),
        "PR_AUC": float(
            average_precision_score(
                y_validation,
                probabilities,
            )
        ),
        "BRIER_SCORE": float(
            brier_score_loss(y_validation, probabilities)
        ),
        "TOP_10_RECALL": float(
            top_delays / total_delays if total_delays else 0.0
        ),
        "TOP_10_LIFT": float(
            top_delay_rate / overall_delay_rate
            if overall_delay_rate else 0.0
        ),
        "DELAY_PRECISION": float(delay_precision),
        "DELAY_RECALL": float(delay_recall),
        "DELAY_F1": float(delay_f1),
    }


def positive_class_weight(labels):
    """Return the negative-to-positive ratio for XGBoost."""
    positive_count = int(np.sum(labels == 1))
    negative_count = int(np.sum(labels == 0))

    if positive_count == 0 or negative_count == 0:
        raise ValueError(
            "Both target classes must be present in every training fold."
        )

    return negative_count / positive_count


def confusion_matrix_table(y_true, y_predicted):
    """Return a clearly labeled binary confusion matrix."""
    matrix = confusion_matrix(
        y_true,
        y_predicted,
        labels=[0, 1],
    )
    return pd.DataFrame(
        matrix,
        index=["Actual On Time (0)", "Actual Delayed (1)"],
        columns=["Predicted On Time (0)", "Predicted Delayed (1)"],
    )


def confusion_count_table(y_true, y_predicted):
    """Return explicit TN, FP, FN, and TP counts with definitions."""
    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_predicted,
        labels=[0, 1],
    ).ravel()

    return pd.DataFrame(
        [
            {
                "CONFUSION_TERM": "TN",
                "MEANING": "Actual on-time flight predicted as on time",
                "COUNT": int(tn),
            },
            {
                "CONFUSION_TERM": "FP",
                "MEANING": "Actual on-time flight predicted as delayed",
                "COUNT": int(fp),
            },
            {
                "CONFUSION_TERM": "FN",
                "MEANING": "Actual delayed flight predicted as on time",
                "COUNT": int(fn),
            },
            {
                "CONFUSION_TERM": "TP",
                "MEANING": "Actual delayed flight predicted as delayed",
                "COUNT": int(tp),
            },
        ]
    )


print("Shared local-matrix and evaluation utilities created.")


## Class-Imbalance Strategy

Delayed flights form the minority class. A classifier can therefore achieve high overall Accuracy by predicting most flights as on time while missing many actual delays.

Class imbalance is handled only within each training dataset:

- Logistic Regression uses `class_weight="balanced"`.
- Random Forest uses `class_weight="balanced_subsample"`.
- XGBoost uses `scale_pos_weight`, calculated as the number of on-time training observations divided by the number of delayed training observations.

Validation observations are not oversampled, undersampled, or synthetically generated. Keeping the validation distribution unchanged produces metrics that reflect realistic flight operations and prevents information from the validation period from influencing model training.

Delayed-flight Recall, delayed-flight F1-score, and PR AUC are reported in addition to overall and weighted metrics. These metrics expose minority-class performance that Accuracy alone may conceal.


## Metric Definitions and Delayed-Class Formulas

The target is binary: `0` represents an on-time flight and `1` represents a delayed flight. Metrics beginning with `DELAY_` evaluate class `1` specifically, whereas the unprefixed Precision, Recall, and F1 columns are weighted averages across both classes.

Let `TP` be delayed flights correctly predicted as delayed, `FN` be delayed flights incorrectly predicted as on time, and `FP` be on-time flights incorrectly predicted as delayed.

**Delayed-flight Recall**

`DELAY_RECALL = TP / (TP + FN)`

This answers: Of all flights that actually became delayed, what proportion did the model detect?

**Delayed-flight Precision**

`DELAY_PRECISION = TP / (TP + FP)`

This answers: Of all flights flagged as delayed, what proportion actually became delayed?

**Delayed-flight F1-score**

`DELAY_F1 = 2 * (DELAY_PRECISION * DELAY_RECALL) / (DELAY_PRECISION + DELAY_RECALL)`

This balances detecting delayed flights against avoiding excessive false delay alerts.

The result-table names have the following meanings:

- `ACCURACY`: proportion of all predictions that are correct.
- `PRECISION`: class-frequency-weighted Precision across classes `0` and `1`.
- `RECALL`: class-frequency-weighted Recall across classes `0` and `1`; in single-label classification it equals Accuracy and is not the delayed-class Recall.
- `F1_SCORE`: class-frequency-weighted F1 across classes `0` and `1`.
- `DELAY_PRECISION`, `DELAY_RECALL`, and `DELAY_F1`: metrics for delayed flights only.
- `ROC_AUC`: discrimination between on-time and delayed flights across thresholds.
- `PR_AUC`: probability-ranking quality for the minority delayed-flight class.

Delayed-flight Recall is emphasized because false negatives are actual delays that receive no warning. It is interpreted together with delayed-flight F1 and PR AUC so that a model is not rewarded merely for flagging nearly every flight as delayed.


## Shared Baseline Dataset

Before hyperparameter tuning, the three standard-Python algorithms are trained and evaluated using the same bounded chronological datasets. Training observations come only from the January–August development period. Validation observations come from the later September–October validation period.

The same feature matrix and labels are reused across all three algorithms, ensuring that baseline differences arise from the algorithms rather than from different samples.


In [0]:
baseline_training_df = bounded_uniform_sample(
    df_train_prepared.select(*MODEL_INPUT_COLUMNS, TARGET_COLUMN),
    LOCAL_TRAIN_MAX_ROWS,
    seed=RANDOM_SEED,
)
baseline_validation_df = bounded_uniform_sample(
    df_validation_prepared.select(*MODEL_INPUT_COLUMNS, TARGET_COLUMN),
    LOCAL_VALIDATION_MAX_ROWS,
    seed=RANDOM_SEED + 1,
)

(
    baseline_preprocessor,
    X_baseline_train,
    y_baseline_train,
    X_baseline_validation,
    y_baseline_validation,
) = prepare_training_validation_matrices(
    baseline_training_df,
    baseline_validation_df,
)

print(f"Shared baseline training rows: {X_baseline_train.shape[0]:,}")
print(
    "Shared baseline validation rows: "
    f"{X_baseline_validation.shape[0]:,}"
)
print(
    "Baseline training delayed-flight rate: "
    f"{np.mean(y_baseline_train):.4f}"
)
print(
    "Baseline validation delayed-flight rate: "
    f"{np.mean(y_baseline_validation):.4f}"
)


## Train and Evaluate Standard-Python Baselines

The baseline configurations provide an initial reference before tuning. Class-imbalance controls are enabled for every trainable candidate, while the majority-class baseline demonstrates why Accuracy alone is insufficient.


In [0]:
majority_class = int(
    np.bincount(y_baseline_train).argmax()
)
majority_predictions = np.full(
    y_baseline_validation.shape,
    majority_class,
    dtype=np.int8,
)

majority_weighted_precision, majority_weighted_recall, majority_weighted_f1, _ = (
    precision_recall_fscore_support(
        y_baseline_validation,
        majority_predictions,
        average="weighted",
        zero_division=0,
    )
)

baseline_rows = [
    {
        "MODEL": "Majority Class Baseline",
        "ACCURACY": float(
            accuracy_score(
                y_baseline_validation,
                majority_predictions,
            )
        ),
        "PRECISION": float(majority_weighted_precision),
        "RECALL": float(majority_weighted_recall),
        "F1_SCORE": float(majority_weighted_f1),
        "ROC_AUC": None,
        "PR_AUC": None,
        "BRIER_SCORE": float(
            brier_score_loss(
                y_baseline_validation,
                majority_predictions.astype(float),
            )
        ),
        "TOP_10_RECALL": 0.0,
        "TOP_10_LIFT": 0.0,
        "DELAY_PRECISION": 0.0,
        "DELAY_RECALL": 0.0,
        "DELAY_F1": 0.0,
    }
]

baseline_estimators = {
    "Logistic Regression (Baseline)": LogisticRegression(
        C=1.0,
        penalty="l2",
        solver="liblinear",
        class_weight="balanced",
        max_iter=500,
        random_state=RANDOM_SEED,
    ),
    "Random Forest (Baseline)": RandomForestClassifier(
        n_estimators=100,
        max_depth=12,
        min_samples_leaf=2,
        max_features="sqrt",
        class_weight="balanced_subsample",
        random_state=RANDOM_SEED,
        n_jobs=-1,
    ),
    "XGBoost (Baseline)": XGBClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.10,
        min_child_weight=1,
        subsample=0.80,
        colsample_bytree=0.80,
        reg_lambda=1.0,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        scale_pos_weight=positive_class_weight(y_baseline_train),
        random_state=RANDOM_SEED,
        n_jobs=-1,
    ),
}

baseline_predictions = {}

for model_name, estimator in baseline_estimators.items():
    estimator.fit(X_baseline_train, y_baseline_train)
    baseline_predictions[model_name] = (
        estimator.predict(X_baseline_validation).astype(np.int8)
    )
    metrics = evaluate_python_classifier(
        estimator,
        X_baseline_validation,
        y_baseline_validation,
    )
    baseline_rows.append(
        {
            "MODEL": model_name,
            **metrics,
        }
    )

model_comparison = spark.createDataFrame(baseline_rows)
display(model_comparison.orderBy("MODEL"))


## Baseline Confusion Matrices

The confusion matrices below show how each untuned algorithm classified the same chronological validation observations. Rows represent actual outcomes and columns represent predicted outcomes.

- The upper-left cell is the number of correctly identified on-time flights (true negatives).
- The upper-right cell is the number of on-time flights incorrectly flagged as delayed (false positives).
- The lower-left cell is the number of delayed flights missed by the model (false negatives).
- The lower-right cell is the number of delayed flights correctly identified (true positives).

For this project, the lower-left cell is especially important because it contains actual delays that would receive no operational warning.


In [0]:
for model_name, predictions in baseline_predictions.items():
    print(f"Baseline confusion matrix: {model_name}")
    display(
        confusion_matrix_table(
            y_baseline_validation,
            predictions,
        )
    )
    print(f"Baseline TN/FP/FN/TP counts: {model_name}")
    display(
        confusion_count_table(
            y_baseline_validation,
            predictions,
        )
    )


## Baseline Comparison Interpretation

The majority-class baseline is included as a diagnostic reference rather than a viable operational model. Its Accuracy can appear strong because most flights are on time, but its delayed-flight Recall is zero.

The three machine-learning baselines use identical local training and validation matrices and explicitly account for class imbalance. Their relative performance therefore provides a fair preliminary comparison. Final selection is not based on this single validation period; it is based on average performance across the shared chronological cross-validation folds below.


## Unified Chronological Cross-Validation

One cross-validation workflow is used for Logistic Regression, Random Forest, and XGBoost.

The January–August development period is divided into five expanding-window folds:

| Fold | Training period | Validation period |
|---|---|---|
| 1 | January–March 2025 | April 2025 |
| 2 | January–April 2025 | May 2025 |
| 3 | January–May 2025 | June 2025 |
| 4 | January–June 2025 | July 2025 |
| 5 | January–July 2025 | August 2025 |

For each fold, the chronological date boundaries are applied before sampling. One reproducible uniform training sample and one reproducible uniform validation sample are then created and converted to local sparse matrices. These exact matrices are reused by every algorithm and every hyperparameter configuration.

This arrangement prevents future observations from entering earlier training periods, keeps validation distributions natural, and avoids algorithm-specific sampling differences.


In [0]:
TUNING_FOLDS = [
    {
        "train_end": "2025-03-31",
        "validation_start": "2025-04-01",
        "validation_end": "2025-04-30",
    },
    {
        "train_end": "2025-04-30",
        "validation_start": "2025-05-01",
        "validation_end": "2025-05-31",
    },
    {
        "train_end": "2025-05-31",
        "validation_start": "2025-06-01",
        "validation_end": "2025-06-30",
    },
    {
        "train_end": "2025-06-30",
        "validation_start": "2025-07-01",
        "validation_end": "2025-07-31",
    },
    {
        "train_end": "2025-07-31",
        "validation_start": "2025-08-01",
        "validation_end": "2025-08-31",
    },
]

df_tuning_complete = (
    df_train_prepared
    .select("FL_DATE", *MODEL_INPUT_COLUMNS, TARGET_COLUMN)
)

LOCAL_CV_FOLDS = []

for fold_number, fold in enumerate(TUNING_FOLDS, start=1):
    complete_training_fold = df_tuning_complete.filter(
        F.col("FL_DATE")
        <= F.to_date(F.lit(fold["train_end"]))
    )
    complete_validation_fold = df_tuning_complete.filter(
        F.col("FL_DATE").between(
            F.to_date(F.lit(fold["validation_start"])),
            F.to_date(F.lit(fold["validation_end"])),
        )
    )

    sampled_training_fold = bounded_uniform_sample(
        complete_training_fold,
        LOCAL_TRAIN_MAX_ROWS,
        seed=RANDOM_SEED + fold_number * 10,
    )
    sampled_validation_fold = bounded_uniform_sample(
        complete_validation_fold,
        LOCAL_VALIDATION_MAX_ROWS,
        seed=RANDOM_SEED + fold_number * 10 + 1,
    )

    (
        fold_preprocessor,
        X_train_fold,
        y_train_fold,
        X_validation_fold,
        y_validation_fold,
    ) = prepare_training_validation_matrices(
        sampled_training_fold,
        sampled_validation_fold,
    )

    LOCAL_CV_FOLDS.append(
        {
            "fold": fold_number,
            "X_train": X_train_fold,
            "y_train": y_train_fold,
            "X_validation": X_validation_fold,
            "y_validation": y_validation_fold,
            "preprocessor": fold_preprocessor,
            "scale_pos_weight": positive_class_weight(
                y_train_fold
            ),
        }
    )

    print(
        f"Fold {fold_number}: "
        f"{X_train_fold.shape[0]:,} training rows, "
        f"{X_validation_fold.shape[0]:,} validation rows, "
        f"training delay rate={np.mean(y_train_fold):.4f}, "
        f"validation delay rate={np.mean(y_validation_fold):.4f}"
    )

print("Shared chronological folds prepared for all algorithms.")


## Two-Stage Hyperparameter Search Spaces

Each algorithm first receives a broad search across model complexity and class-imbalance treatment. A focused refinement stage is then generated around the broad-search configuration with the highest mean PR AUC. All candidates are evaluated across the same five chronological folds.

The search spaces cover:

- Logistic Regression regularization strength and penalty type.
- Random Forest ensemble size, tree depth, leaf size, and feature subsampling.
- XGBoost boosting rounds, learning rate, tree depth, child-weight control, row and feature subsampling, and L2 regularization.

The imbalance strategies compared are the natural training distribution, estimator class weighting, and training-only random undersampling. Validation samples remain unchanged. Runtime is not artificially extended and will depend on Databricks capacity.


In [0]:
LOGISTIC_REGRESSION_PARAMETER_GRID = [
    {"C": 0.01, "penalty": "l2", "solver": "liblinear"},
    {"C": 0.10, "penalty": "l2", "solver": "liblinear"},
    {"C": 1.00, "penalty": "l2", "solver": "liblinear"},
    {"C": 10.0, "penalty": "l2", "solver": "liblinear"},
    {"C": 0.01, "penalty": "l1", "solver": "liblinear"},
    {"C": 0.10, "penalty": "l1", "solver": "liblinear"},
    {"C": 1.00, "penalty": "l1", "solver": "liblinear"},
    {"C": 10.0, "penalty": "l1", "solver": "liblinear"},
]

RANDOM_FOREST_PARAMETER_GRID = [
    {
        "n_estimators": 100,
        "max_depth": 8,
        "min_samples_leaf": 1,
        "max_features": "sqrt",
    },
    {
        "n_estimators": 100,
        "max_depth": 12,
        "min_samples_leaf": 2,
        "max_features": "sqrt",
    },
    {
        "n_estimators": 200,
        "max_depth": 8,
        "min_samples_leaf": 2,
        "max_features": "sqrt",
    },
    {
        "n_estimators": 200,
        "max_depth": 12,
        "min_samples_leaf": 5,
        "max_features": "sqrt",
    },
    {
        "n_estimators": 300,
        "max_depth": 12,
        "min_samples_leaf": 2,
        "max_features": "log2",
    },
    {
        "n_estimators": 300,
        "max_depth": 16,
        "min_samples_leaf": 5,
        "max_features": "sqrt",
    },
    {
        "n_estimators": 200,
        "max_depth": None,
        "min_samples_leaf": 5,
        "max_features": "sqrt",
    },
    {
        "n_estimators": 300,
        "max_depth": None,
        "min_samples_leaf": 10,
        "max_features": "log2",
    },
]

XGBOOST_PARAMETER_GRID = [
    {
        "n_estimators": 100,
        "max_depth": 4,
        "learning_rate": 0.10,
        "min_child_weight": 1,
        "subsample": 0.80,
        "colsample_bytree": 0.80,
        "reg_lambda": 1.0,
    },
    {
        "n_estimators": 200,
        "max_depth": 4,
        "learning_rate": 0.05,
        "min_child_weight": 1,
        "subsample": 0.80,
        "colsample_bytree": 0.80,
        "reg_lambda": 1.0,
    },
    {
        "n_estimators": 300,
        "max_depth": 4,
        "learning_rate": 0.03,
        "min_child_weight": 3,
        "subsample": 0.90,
        "colsample_bytree": 0.80,
        "reg_lambda": 1.0,
    },
    {
        "n_estimators": 100,
        "max_depth": 6,
        "learning_rate": 0.10,
        "min_child_weight": 1,
        "subsample": 0.80,
        "colsample_bytree": 0.80,
        "reg_lambda": 1.0,
    },
    {
        "n_estimators": 200,
        "max_depth": 6,
        "learning_rate": 0.05,
        "min_child_weight": 3,
        "subsample": 0.80,
        "colsample_bytree": 0.90,
        "reg_lambda": 2.0,
    },
    {
        "n_estimators": 300,
        "max_depth": 6,
        "learning_rate": 0.03,
        "min_child_weight": 5,
        "subsample": 0.90,
        "colsample_bytree": 0.90,
        "reg_lambda": 2.0,
    },
    {
        "n_estimators": 200,
        "max_depth": 8,
        "learning_rate": 0.05,
        "min_child_weight": 5,
        "subsample": 0.80,
        "colsample_bytree": 0.80,
        "reg_lambda": 5.0,
    },
    {
        "n_estimators": 300,
        "max_depth": 8,
        "learning_rate": 0.03,
        "min_child_weight": 7,
        "subsample": 0.90,
        "colsample_bytree": 0.80,
        "reg_lambda": 5.0,
    },
]

LOGISTIC_REGRESSION_BROAD_GRID = list(ParameterGrid([
    {
        "C": [0.01, 0.10, 1.0, 10.0],
        "penalty": ["l1"],
        "solver": ["liblinear"],
        "imbalance_strategy": ["natural", "class_weight", "undersample"],
    },
    {
        "C": [0.01, 0.10, 1.0, 10.0],
        "penalty": ["l2"],
        "solver": ["lbfgs"],
        "imbalance_strategy": ["natural", "class_weight", "undersample"],
    },
]))

RANDOM_FOREST_BASE_CONFIGURATIONS = [
    {
        "n_estimators": 100,
        "max_depth": 8,
        "min_samples_leaf": 1,
        "max_features": "sqrt",
        "criterion": "gini",
    },
    {
        "n_estimators": 200,
        "max_depth": 12,
        "min_samples_leaf": 2,
        "max_features": "sqrt",
        "criterion": "gini",
    },
    {
        "n_estimators": 300,
        "max_depth": 16,
        "min_samples_leaf": 5,
        "max_features": "log2",
        "criterion": "entropy",
    },
]

RANDOM_FOREST_BROAD_GRID = [
    {
        **configuration,
        "imbalance_strategy": imbalance_strategy,
    }
    for configuration in RANDOM_FOREST_BASE_CONFIGURATIONS
    for imbalance_strategy in [
        "natural",
        "class_weight",
        "undersample",
    ]
]

XGBOOST_BASE_CONFIGURATIONS = [
    {
        "n_estimators": 150,
        "max_depth": 4,
        "learning_rate": 0.08,
        "min_child_weight": 1,
        "subsample": 0.85,
        "colsample_bytree": 0.85,
        "reg_lambda": 1.0,
        "gamma": 0.0,
    },
    {
        "n_estimators": 250,
        "max_depth": 6,
        "learning_rate": 0.05,
        "min_child_weight": 3,
        "subsample": 0.85,
        "colsample_bytree": 0.85,
        "reg_lambda": 2.0,
        "gamma": 0.0,
    },
    {
        "n_estimators": 300,
        "max_depth": 8,
        "learning_rate": 0.03,
        "min_child_weight": 5,
        "subsample": 0.90,
        "colsample_bytree": 0.80,
        "reg_lambda": 5.0,
        "gamma": 0.5,
    },
]

XGBOOST_BROAD_GRID = [
    {
        **configuration,
        "imbalance_strategy": imbalance_strategy,
    }
    for configuration in XGBOOST_BASE_CONFIGURATIONS
    for imbalance_strategy in [
        "natural",
        "class_weight",
        "undersample",
    ]
]

print(
    "Broad-search configurations: "
    f"LR={len(LOGISTIC_REGRESSION_BROAD_GRID)}, "
    f"RF={len(RANDOM_FOREST_BROAD_GRID)}, "
    f"XGBoost={len(XGBOOST_BROAD_GRID)}"
)
print(
    "Broad-search chronological fits: "
    f"{(len(LOGISTIC_REGRESSION_BROAD_GRID) + len(RANDOM_FOREST_BROAD_GRID) + len(XGBOOST_BROAD_GRID)) * len(TUNING_FOLDS)}"
)


## Shared Estimator Builders

Each builder creates a fresh standard-Python classifier. The `imbalance_strategy` field is evaluated as part of tuning rather than fixed in advance. Training-only undersampling is performed reproducibly; validation data is never rebalanced.


In [0]:
def build_logistic_regression(params, *, scale_pos_weight=None):
    params = dict(params)
    imbalance_strategy = params.pop("imbalance_strategy")
    return LogisticRegression(
        **params,
        class_weight=(
            "balanced" if imbalance_strategy == "class_weight" else None
        ),
        max_iter=1_000,
        random_state=RANDOM_SEED,
    )


def build_random_forest(params, *, scale_pos_weight=None):
    params = dict(params)
    imbalance_strategy = params.pop("imbalance_strategy")
    return RandomForestClassifier(
        **params,
        class_weight=(
            "balanced_subsample"
            if imbalance_strategy == "class_weight"
            else None
        ),
        random_state=RANDOM_SEED,
        n_jobs=-1,
    )


def build_xgboost(params, *, scale_pos_weight):
    params = dict(params)
    imbalance_strategy = params.pop("imbalance_strategy")
    return XGBClassifier(
        **params,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        scale_pos_weight=(
            scale_pos_weight
            if imbalance_strategy == "class_weight"
            else 1.0
        ),
        random_state=RANDOM_SEED,
        n_jobs=-1,
    )


print("Shared standard-Python estimator builders created.")


def training_data_for_strategy(X_train, y_train, strategy, *, seed):
    # Apply a reproducible imbalance treatment to training data only.
    if strategy in {"natural", "class_weight"}:
        return X_train, y_train
    if strategy != "undersample":
        raise ValueError(f"Unknown imbalance strategy: {strategy}")

    positive_indices = np.flatnonzero(y_train == 1)
    negative_indices = np.flatnonzero(y_train == 0)
    if len(positive_indices) == 0 or len(negative_indices) == 0:
        raise ValueError("Both classes are required for undersampling.")

    generator = np.random.default_rng(seed)
    retained_negative_indices = generator.choice(
        negative_indices,
        size=min(len(negative_indices), len(positive_indices)),
        replace=False,
    )
    retained_indices = np.concatenate(
        [positive_indices, retained_negative_indices]
    )
    generator.shuffle(retained_indices)
    return X_train[retained_indices], y_train[retained_indices]


def focused_refinement_grid(model_name, best_params):
    # Build a compact second-stage search around the broad-search winner.
    best_params = dict(best_params)
    strategy = best_params["imbalance_strategy"]

    if model_name == "Logistic Regression":
        return [
            {
                **best_params,
                "C": max(1e-4, best_params["C"] * multiplier),
                "imbalance_strategy": candidate_strategy,
            }
            for multiplier in [0.5, 0.75, 1.0, 1.5, 2.0]
            for candidate_strategy in [
                strategy,
                "natural",
                "class_weight",
                "undersample",
            ]
        ]

    if model_name == "Random Forest":
        best_tree_count = best_params["n_estimators"]
        best_depth = best_params["max_depth"]
        best_leaf_size = best_params["min_samples_leaf"]

        if best_depth is None:
            nearby_depths = [16, 24]
        else:
            nearby_depths = [
                max(4, best_depth - 4),
                best_depth + 4,
            ]

        nearby_tree_counts = [
            max(100, best_tree_count - 100),
            min(300, best_tree_count + 100),
        ]

        refinement_candidates = []
        for tree_count, depth in zip(
            nearby_tree_counts,
            nearby_depths,
        ):
            refinement_candidates.append(
                {
                    **best_params,
                    "n_estimators": tree_count,
                    "max_depth": depth,
                    "min_samples_leaf": max(1, best_leaf_size),
                }
            )

        return refinement_candidates

    if model_name == "XGBoost":
        lower_rounds = max(100, best_params["n_estimators"] - 50)
        higher_rounds = min(400, best_params["n_estimators"] + 50)
        lower_depth = max(3, best_params["max_depth"] - 1)
        higher_depth = min(9, best_params["max_depth"] + 1)

        return [
            {
                **best_params,
                "n_estimators": lower_rounds,
                "max_depth": lower_depth,
                "learning_rate": min(
                    0.15,
                    best_params["learning_rate"] * 1.20,
                ),
            },
            {
                **best_params,
                "n_estimators": higher_rounds,
                "max_depth": higher_depth,
                "learning_rate": max(
                    0.01,
                    best_params["learning_rate"] * 0.80,
                ),
            },
            {
                **best_params,
                "n_estimators": best_params["n_estimators"],
                "min_child_weight": best_params["min_child_weight"] + 2,
                "reg_lambda": best_params["reg_lambda"] * 2.0,
                "gamma": 0.5,
                "subsample": 0.90,
                "colsample_bytree": 0.90,
            },
        ]

    raise ValueError(f"Unsupported model for refinement: {model_name}")


## Shared Hyperparameter-Tuning Function

The same function evaluates every algorithm. For each candidate configuration, it:

1. Uses the same five precomputed chronological folds.
2. Fits only on the fold's earlier training observations.
3. Applies training-only class-imbalance controls.
4. Evaluates on the unchanged later validation observations.
5. Reports the mean and standard deviation across folds.

Mean PR AUC guides tuning because delayed flights are the minority class. Brier Score, top-10% Recall, top-10% Lift, delayed-class metrics, ROC AUC, stability, and cost remain visible for multi-criteria comparison.


In [0]:
RESULT_COLUMNS = [
    "MODEL",
    "SEARCH_STAGE",
    "PARAMETERS",
    "ACCURACY",
    "PRECISION",
    "RECALL",
    "F1_SCORE",
    "ROC_AUC",
    "PR_AUC",
    "BRIER_SCORE",
    "TOP_10_RECALL",
    "TOP_10_LIFT",
    "DELAY_PRECISION",
    "DELAY_RECALL",
    "DELAY_F1",
    "TRAINING_SECONDS",
    "PR_AUC_STD",
    "DELAY_RECALL_STD",
    "DELAY_F1_STD",
    "BRIER_SCORE_STD",
]


def tune_python_classifier(
    model_name,
    estimator_builder,
    parameter_grid,
    *,
    search_stage,
    local_folds=LOCAL_CV_FOLDS,
):
    """Tune one standard-Python classifier on shared time folds."""
    if not parameter_grid:
        raise ValueError(f"{model_name} parameter grid is empty.")

    result_rows = []

    for configuration_number, params in enumerate(
        parameter_grid,
        start=1,
    ):
        fold_metrics = []

        print(
            f"Evaluating {model_name} configuration "
            f"{configuration_number}/{len(parameter_grid)}: {params}"
        )

        for fold in local_folds:
            strategy = params["imbalance_strategy"]
            X_fit, y_fit = training_data_for_strategy(
                fold["X_train"],
                fold["y_train"],
                strategy,
                seed=RANDOM_SEED + fold["fold"] * 100 + configuration_number,
            )
            estimator = estimator_builder(
                params,
                scale_pos_weight=fold["scale_pos_weight"],
            )

            training_start = time.perf_counter()
            estimator.fit(
                X_fit,
                y_fit,
            )
            training_seconds = time.perf_counter() - training_start

            metrics = evaluate_python_classifier(
                estimator,
                fold["X_validation"],
                fold["y_validation"],
            )
            metrics["TRAINING_SECONDS"] = float(training_seconds)
            fold_metrics.append(metrics)

            print(
                f"  Fold {fold['fold']}: "
                f"delay recall={metrics['DELAY_RECALL']:.4f}, "
                f"delay F1={metrics['DELAY_F1']:.4f}, "
                f"PR AUC={metrics['PR_AUC']:.4f}"
            )

        mean_metric_names = [
            "ACCURACY", "PRECISION", "RECALL", "F1_SCORE",
            "ROC_AUC", "PR_AUC", "BRIER_SCORE", "TOP_10_RECALL",
            "TOP_10_LIFT", "DELAY_PRECISION", "DELAY_RECALL",
            "DELAY_F1", "TRAINING_SECONDS",
        ]
        averaged = {
            metric_name: round(
                float(
                    np.mean(
                        [
                            row[metric_name]
                            for row in fold_metrics
                        ]
                    )
                ),
                2 if metric_name == "TRAINING_SECONDS" else 4,
            )
            for metric_name in mean_metric_names
        }
        stability = {
            "PR_AUC_STD": round(float(np.std(
                [row["PR_AUC"] for row in fold_metrics], ddof=0
            )), 4),
            "DELAY_RECALL_STD": round(float(np.std(
                [row["DELAY_RECALL"] for row in fold_metrics], ddof=0
            )), 4),
            "DELAY_F1_STD": round(float(np.std(
                [row["DELAY_F1"] for row in fold_metrics], ddof=0
            )), 4),
            "BRIER_SCORE_STD": round(float(np.std(
                [row["BRIER_SCORE"] for row in fold_metrics], ddof=0
            )), 4),
        }

        result_rows.append(
            {
                "MODEL": model_name,
                "SEARCH_STAGE": search_stage,
                "PARAMETERS": str(params),
                **averaged,
                **stability,
            }
        )

    return (
        spark.createDataFrame(result_rows)
        .select(*RESULT_COLUMNS)
        .orderBy(
            F.desc("PR_AUC"),
            F.asc("PR_AUC_STD"),
            F.desc("DELAY_F1"),
            F.desc("DELAY_RECALL"),
            F.asc("BRIER_SCORE"),
            F.desc("TOP_10_LIFT"),
            F.desc("ROC_AUC"),
            F.asc("TRAINING_SECONDS"),
        )
    )


def rank_tuning_results(results):
    # Apply the shared PR-AUC-led ranking to tuning results.
    return results.orderBy(
        F.desc("PR_AUC"),
        F.asc("PR_AUC_STD"),
        F.desc("DELAY_F1"),
        F.desc("DELAY_RECALL"),
        F.desc("DELAY_PRECISION"),
        F.asc("BRIER_SCORE"),
        F.desc("TOP_10_LIFT"),
        F.desc("ROC_AUC"),
        F.asc("TRAINING_SECONDS"),
    )


def run_two_stage_search(model_name, estimator_builder, broad_grid):
    # Run broad search, then refine around the leading configuration.
    broad_results = tune_python_classifier(
        model_name=model_name,
        estimator_builder=estimator_builder,
        parameter_grid=broad_grid,
        search_stage="broad",
    )
    broad_winner = broad_results.first()
    if broad_winner is None:
        raise ValueError(f"Broad search returned no result for {model_name}.")

    focused_grid = focused_refinement_grid(
        model_name,
        ast.literal_eval(broad_winner["PARAMETERS"]),
    )
    # Keep the refinement meaningful but feasible on Free Edition.
    focused_limits = {
        "Logistic Regression": 12,
        "Random Forest": 2,
        "XGBoost": 3,
    }
    focused_limit = focused_limits[model_name]
    focused_grid = focused_grid[:focused_limit]

    focused_results = tune_python_classifier(
        model_name=model_name,
        estimator_builder=estimator_builder,
        parameter_grid=focused_grid,
        search_stage="refinement",
    )
    return rank_tuning_results(
        broad_results.unionByName(focused_results)
    )


print("Unified standard-Python tuning function created.")


## Tune Logistic Regression

Logistic Regression uses a two-stage search across all five chronological folds. The broad stage compares regularization, penalty, solver, and class-imbalance strategies. A focused stage then explores nearby settings around the broad-stage leader. Mean PR AUC guides the search, with fold-to-fold stability and delayed-class metrics used as supporting evidence.


In [0]:
lr_tuning_results = run_two_stage_search(
    model_name="Logistic Regression",
    estimator_builder=build_logistic_regression,
    broad_grid=LOGISTIC_REGRESSION_BROAD_GRID,
)

display(lr_tuning_results)


## Tune Random Forest

Random Forest uses the same two-stage, five-fold workflow. The search compares tree count, depth, minimum leaf size, feature subsampling, split criterion, and class-imbalance strategy. Validation data retain their natural class distribution.


In [0]:
rf_tuning_results = run_two_stage_search(
    model_name="Random Forest",
    estimator_builder=build_random_forest,
    broad_grid=RANDOM_FOREST_BROAD_GRID,
)

display(rf_tuning_results)


## Tune XGBoost

XGBoost uses the same two-stage, five-fold workflow. The search compares boosting rounds, depth, learning rate, child-weight, row and feature subsampling, regularization, minimum loss reduction, and class-imbalance strategy. Any class weight is calculated only from the current training fold.


In [0]:
xgb_tuning_results = run_two_stage_search(
    model_name="XGBoost",
    estimator_builder=build_xgboost,
    broad_grid=XGBOOST_BROAD_GRID,
)

display(xgb_tuning_results)


## Tuned Model Comparison

Hyperparameter tuning and final algorithm selection use related but distinct rules. Within each algorithm, mean PR AUC across the five folds guides selection of its strongest configuration. The within-algorithm tie-breakers are:

1. Highest mean PR AUC across the five folds
2. Lowest PR AUC standard deviation
3. Highest delayed-flight F1-score, Recall, and Precision
4. Lowest Brier Score
5. Highest Recall and Lift within the top 10% highest-risk flights
6. Highest ROC AUC and lowest average training time

PR AUC leads hyperparameter tuning because delayed flights are the minority class and the metric evaluates the Precision–Recall trade-off across probability thresholds. It does **not** automatically select the final algorithm.

Final algorithm selection uses a transparent weighted score combining PR AUC, ROC AUC, delayed-flight Recall, delayed-flight Precision, delayed-flight F1-score, Brier Score, top-10% Recall and Lift, temporal stability, interpretability, and computational cost. `DELAY_RECALL` is therefore included explicitly without being allowed to dominate the entire decision.

Delayed-flight Recall is calculated as `TP / (TP + FN)`. It answers: *Of all flights that actually became delayed, what proportion did the model identify?* It is operationally important because a missed delayed flight cannot be prioritized before departure.

Recall is not sufficient by itself. A model could obtain high Recall by flagging too many on-time flights as delayed. Delayed-flight F1-score therefore checks the balance between Recall and Precision, while PR AUC evaluates how effectively the model ranks the minority delayed-flight class across decision thresholds. If the project later defines an explicit acceptable missed-delay rate, an even stronger policy would be to require that minimum Recall first and then select the eligible model with the highest delayed-flight F1-score and PR AUC.


In [0]:
def best_tuned_configuration(results):
    return rank_tuning_results(results).limit(1)


best_logistic_regression = best_tuned_configuration(
    lr_tuning_results
)
best_random_forest = best_tuned_configuration(
    rf_tuning_results
)
best_xgboost = best_tuned_configuration(
    xgb_tuning_results
)

tuned_model_comparison = (
    best_logistic_regression
    .unionByName(best_random_forest)
    .unionByName(best_xgboost)
)


def normalized_candidate_values(rows, column, *, higher_is_better=True):
    values = [float(row[column]) for row in rows]
    minimum = min(values)
    maximum = max(values)
    if maximum == minimum:
        return [0.5 for _ in values]
    normalized = [
        (value - minimum) / (maximum - minimum)
        for value in values
    ]
    if not higher_is_better:
        normalized = [1.0 - value for value in normalized]
    return normalized


# Mean PR AUC selects the best hyperparameters within each algorithm.
# Final algorithm selection is broader and combines predictive,
# operational, calibration, stability, interpretability, and cost evidence.
candidate_rows = [row.asDict() for row in tuned_model_comparison.collect()]

selection_components = {
    "PR_AUC_COMPONENT": normalized_candidate_values(candidate_rows, "PR_AUC"),
    "ROC_AUC_COMPONENT": normalized_candidate_values(candidate_rows, "ROC_AUC"),
    "DELAY_RECALL_COMPONENT": normalized_candidate_values(
        candidate_rows, "DELAY_RECALL"
    ),
    "DELAY_PRECISION_COMPONENT": normalized_candidate_values(
        candidate_rows, "DELAY_PRECISION"
    ),
    "DELAY_F1_COMPONENT": normalized_candidate_values(candidate_rows, "DELAY_F1"),
    "CALIBRATION_COMPONENT": normalized_candidate_values(
        candidate_rows, "BRIER_SCORE", higher_is_better=False
    ),
    "TOP_10_RECALL_COMPONENT": normalized_candidate_values(
        candidate_rows, "TOP_10_RECALL"
    ),
    "TOP_10_LIFT_COMPONENT": normalized_candidate_values(
        candidate_rows, "TOP_10_LIFT"
    ),
    "STABILITY_COMPONENT": normalized_candidate_values(
        candidate_rows, "PR_AUC_STD", higher_is_better=False
    ),
    "COST_COMPONENT": normalized_candidate_values(
        candidate_rows, "TRAINING_SECONDS", higher_is_better=False
    ),
}

# Transparent qualitative scores: linear Logistic Regression is easiest
# to explain, while ensemble tree models require additional explanation tools.
interpretability_scores = {
    "Logistic Regression": 1.00,
    "Random Forest": 0.70,
    "XGBoost": 0.60,
}

selection_weights = {
    "PR_AUC_COMPONENT": 0.10,
    "ROC_AUC_COMPONENT": 0.05,
    "DELAY_RECALL_COMPONENT": 0.15,
    "DELAY_PRECISION_COMPONENT": 0.10,
    "DELAY_F1_COMPONENT": 0.15,
    "CALIBRATION_COMPONENT": 0.15,
    "TOP_10_RECALL_COMPONENT": 0.075,
    "TOP_10_LIFT_COMPONENT": 0.075,
    "STABILITY_COMPONENT": 0.05,
    "INTERPRETABILITY_SCORE": 0.05,
    "COST_COMPONENT": 0.05,
}

scored_candidate_rows = []
for row_index, row in enumerate(candidate_rows):
    scored_row = dict(row)
    for component_name, component_values in selection_components.items():
        scored_row[component_name] = float(component_values[row_index])
    scored_row["INTERPRETABILITY_SCORE"] = float(
        interpretability_scores[row["MODEL"]]
    )
    scored_row["SELECTION_SCORE"] = round(
        sum(
            selection_weights[component_name]
            * scored_row[component_name]
            for component_name in selection_weights
        ),
        4,
    )
    scored_candidate_rows.append(scored_row)

ranked_model_comparison = (
    spark.createDataFrame(scored_candidate_rows)
    .orderBy(
        F.desc("SELECTION_SCORE"),
        F.desc("PR_AUC"),
        F.desc("DELAY_F1"),
        F.desc("DELAY_RECALL"),
        F.asc("BRIER_SCORE"),
        F.asc("TRAINING_SECONDS"),
    )
)

display(ranked_model_comparison)
print("Final multi-criteria selection weights:")
for criterion, weight in selection_weights.items():
    print(f"  {criterion}: {weight:.1%}")


## Tuned Confusion Matrices Across Chronological Folds

For each algorithm, its strongest hyperparameter configuration is refitted separately within each of the five expanding-window folds. Predictions from the five validation months are pooled into one confusion matrix. A calibration curve is also shown for each model to compare predicted risk with observed delay frequency.

These matrices provide an interpretable after-tuning comparison on identical chronological validation observations. They are diagnostic cross-validation summaries, not final holdout results. Notebook 08 remains responsible for definitive evaluation on later untouched data.


In [0]:
best_parameter_rows = {
    "Logistic Regression": best_logistic_regression.first(),
    "Random Forest": best_random_forest.first(),
    "XGBoost": best_xgboost.first(),
}

estimator_builders = {
    "Logistic Regression": build_logistic_regression,
    "Random Forest": build_random_forest,
    "XGBoost": build_xgboost,
}

tuned_confusion_matrices = {}
tuned_probability_diagnostics = {}

for model_name, result_row in best_parameter_rows.items():
    if result_row is None:
        raise ValueError(f"No tuned result found for {model_name}.")

    best_parameters = ast.literal_eval(result_row["PARAMETERS"])
    pooled_actual = []
    pooled_predicted = []
    pooled_probability = []

    for fold in LOCAL_CV_FOLDS:
        estimator = estimator_builders[model_name](
            best_parameters,
            scale_pos_weight=fold["scale_pos_weight"],
        )
        fit_X, fit_y = training_data_for_strategy(
            fold["X_train"],
            fold["y_train"],
            best_parameters["imbalance_strategy"],
            seed=RANDOM_SEED + fold["fold"] * 100,
        )
        estimator.fit(fit_X, fit_y)

        pooled_actual.append(fold["y_validation"])
        fold_probability = estimator.predict_proba(fold["X_validation"])[:, 1]
        pooled_probability.append(fold_probability)
        pooled_predicted.append((fold_probability >= 0.5).astype(np.int8))

    pooled_actual = np.concatenate(pooled_actual)
    pooled_predicted = np.concatenate(pooled_predicted)
    pooled_probability = np.concatenate(pooled_probability)

    tuned_confusion_matrices[model_name] = confusion_matrix_table(
        pooled_actual,
        pooled_predicted,
    )

    print(f"Tuned chronological confusion matrix: {model_name}")
    display(tuned_confusion_matrices[model_name])
    print(f"Tuned chronological TN/FP/FN/TP counts: {model_name}")
    display(
        confusion_count_table(
            pooled_actual,
            pooled_predicted,
        )
    )

    probability_true, probability_predicted = calibration_curve(
        pooled_actual,
        pooled_probability,
        n_bins=10,
        strategy="quantile",
    )
    tuned_probability_diagnostics[model_name] = {
        "actual": pooled_actual,
        "probability": pooled_probability,
        "brier_score": brier_score_loss(pooled_actual, pooled_probability),
    }
    plt.plot(
        probability_predicted,
        probability_true,
        marker="o",
        label=f"{model_name} (Brier={tuned_probability_diagnostics[model_name]['brier_score']:.4f})",
    )

plt.plot([0, 1], [0, 1], linestyle="--", color="black", label="Perfect calibration")
plt.xlabel("Mean predicted delay probability")
plt.ylabel("Observed delayed-flight rate")
plt.title("Chronological Cross-Validation Calibration Curves")
plt.legend()
plt.grid(alpha=0.25)
plt.show()


## Select the Candidate Final Model

The candidate model is selected dynamically from the ranked comparison and is not hard-coded. Every algorithm uses the same five chronological periods, training and validation caps, validation observations, imbalance alternatives, and metrics. This controls the evaluation conditions and reduces model-comparison bias.

The final `SELECTION_SCORE` combines delayed-flight detection (`DELAY_RECALL`, Precision, and F1), minority-class ranking (PR AUC and ROC AUC), probability calibration (inverse Brier Score), operational usefulness (top-10% Recall and Lift), temporal stability, interpretability, and computational cost. The comparison cell prints every component weight so the decision is auditable. No single metric determines the final candidate by itself.

The selected algorithm and its standard-Python hyperparameters are saved for downstream evaluation.


In [0]:
selected_model_row = ranked_model_comparison.first()

if selected_model_row is None:
    raise ValueError(
        "The ranked tuned-model comparison contains no results."
    )

SELECTED_MODEL_NAME = selected_model_row["MODEL"]
SELECTED_MODEL_PARAMETERS = ast.literal_eval(
    selected_model_row["PARAMETERS"]
)

selected_model_summary = (
    ranked_model_comparison
    .filter(F.col("MODEL") == SELECTED_MODEL_NAME)
    .limit(1)
)

display(selected_model_summary)
print(f"Selected model: {SELECTED_MODEL_NAME}")
print(f"Selected parameters: {SELECTED_MODEL_PARAMETERS}")
print(
    "Selection policy: mean PR AUC, temporal stability, delayed-class "
    "F1/Recall/Precision, Brier Score, top-10% lift/recall, ROC AUC, "
    "interpretability, and computational cost."
)


In [0]:
test

## Candidate-Selection Interpretation

The displayed results determine the selected candidate. The interpretation should be written from the newly generated table after the revised notebook has run; earlier numerical conclusions from the Spark-based workflow must not be reused.

The selected candidate must still be retrained using its chosen standard-Python hyperparameters and evaluated on the untouched final holdout period in Notebook 08. The final holdout results—not the tuning-fold averages—provide the definitive estimate of future predictive performance.


## Persist Modeling Checkpoints

This section saves the leakage-safe raw modeling tables, tuned-model comparison, preprocessing specification, and selected-candidate metadata required by downstream notebooks.

Feature hashing is no longer used. The preprocessing manifest records the ordered raw inputs and the scikit-learn `ColumnTransformer` design. Notebook 08 must fit a fresh preprocessor on its training period, transform validation and holdout data with that fitted transformer, reconstruct the selected estimator, and perform final calibration and holdout evaluation.


In [0]:
import json

checkpoint_tables = [
    (cfg.MODELING_TRAIN_HIST_TABLE, df_train_hist),
    (cfg.MODELING_VALIDATION_HIST_TABLE, df_validation_hist),
    (cfg.MODELING_TEST_HIST_TABLE, df_test_hist),
]

for table_name, dataframe in checkpoint_tables:
    row_count = dataframe.count()
    print(f"Saving {table_name}: {row_count:,} rows")
    (
        dataframe.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table_name)
    )

(
    tuned_model_comparison.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(cfg.TUNED_MODEL_COMPARISON_TABLE)
)

feature_manifest = {
    "model_input_columns": MODEL_INPUT_COLUMNS,
    "categorical_columns": CATEGORICAL_COLUMNS,
    "numerical_columns": NUMERICAL_COLUMNS,
    "target_column": TARGET_COLUMN,
    "preprocessing_library": "scikit-learn",
    "preprocessing_transformer": "ColumnTransformer",
    "categorical_transform": "most-frequent imputation plus OneHotEncoder(handle_unknown='ignore')",
    "numerical_transform": "median imputation plus StandardScaler(with_mean=False)",
    "fit_boundary": "fit on training only within each chronological fold",
    "selected_model_name": SELECTED_MODEL_NAME,
    "selected_model_parameters": SELECTED_MODEL_PARAMETERS,
}

candidate_selection = {
    "selected_model_name": SELECTED_MODEL_NAME,
    "selected_model_parameters": SELECTED_MODEL_PARAMETERS,
    "selection_source": "Five-fold chronological multi-criteria comparison",
    "selected_after_hyperparameter_tuning": True,
    "modeling_implementation": "standard_python",
    "cross_validation": "five expanding chronological folds",
    "class_imbalance_strategy": "natural, class-weighted, and training-only undersampled alternatives",
    "selection_policy": "weighted multi-criteria score including delayed recall, precision, F1, calibration, top-risk usefulness, stability, interpretability, and cost",
}

dbutils.fs.put(
    cfg.MODEL_FEATURE_MANIFEST_PATH,
    json.dumps(feature_manifest, indent=4),
    overwrite=True,
)
dbutils.fs.put(
    cfg.CANDIDATE_SELECTION_PATH,
    json.dumps(candidate_selection, indent=4),
    overwrite=True,
)

print("Modeling checkpoints saved successfully.")
print(f"Preprocessing manifest: {cfg.MODEL_FEATURE_MANIFEST_PATH}")
print(f"Candidate selection: {cfg.CANDIDATE_SELECTION_PATH}")


In [0]:
candidate_selection_saved = json.loads(
    dbutils.fs.head(cfg.CANDIDATE_SELECTION_PATH, 10_000)
)
print(json.dumps(candidate_selection_saved, indent=4))

assert candidate_selection_saved["selected_model_name"] == SELECTED_MODEL_NAME
assert (
    candidate_selection_saved["selected_model_parameters"]
    == SELECTED_MODEL_PARAMETERS
)
print("Candidate-selection metadata verified.")
